In [5]:
# Cell 1: Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, check=True,
                            capture_output=True, text=True)
    print(result.stdout)

# Clone FastReID
run("git clone https://github.com/JDAI-CV/fast-reid.git /content/fast-reid")
sys.path.insert(0, '/content/fast-reid')

# Install dependencies
run("pip install yacs termcolor tabulate easydict prettytable")
run("pip install faiss-gpu 2>/dev/null || pip install faiss-cpu")

print("✅ Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.8 MB/s eta 0:00:00

✅ Setup complete


In [6]:
# Cell 2: Verify datasets are accessible directly from Drive
import os
from pathlib import Path

DRIVE_ROOT  = "/content/drive/MyDrive/reid_project"
MARKET_ROOT = f"{DRIVE_ROOT}/Market-1501-v15.09.15"
DUKE_ROOT   = f"{DRIVE_ROOT}/DukeMTMC-reID"

def verify_dataset(root, name):
    folders = ["bounding_box_train", "bounding_box_test", "query"]
    print(f"\n[{name}] Checking: {root}")
    ok = True
    for f in folders:
        p = Path(root) / f
        if p.exists():
            count = len(list(p.glob("*.jpg")))
            print(f"  ✓ {f}: {count} images")
        else:
            print(f"  ✗ MISSING: {f}")
            ok = False
    return ok

m_ok = verify_dataset(MARKET_ROOT, "Market-1501")
d_ok = verify_dataset(DUKE_ROOT, "DukeMTMC-reID")

if m_ok and d_ok:
    print("\n✅ Both datasets verified on Drive")
else:
    print("\n❌ Dataset missing — check folder names in MyDrive/reid_project/")


[Market-1501] Checking: /content/drive/MyDrive/reid_project/Market-1501-v15.09.15
  ✓ bounding_box_train: 12936 images
  ✓ bounding_box_test: 19732 images
  ✓ query: 3368 images

[DukeMTMC-reID] Checking: /content/drive/MyDrive/reid_project/DukeMTMC-reID
  ✓ bounding_box_train: 16522 images
  ✓ bounding_box_test: 17661 images
  ✓ query: 2228 images

✅ Both datasets verified on Drive


In [52]:
import shutil, os

# Copy datasets from Drive to local Colab storage
LOCAL_DATA = "/content/datasets/Market-1501"
DRIVE_MARKET = "/content/drive/MyDrive/reid_project/Market-1501-v15.09.15"

if not os.path.exists(f"{LOCAL_DATA}/Market-1501-v15.09.15"):
    print("Copying Market-1501 from Drive to local... (this takes a few minutes)")
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copytree(DRIVE_MARKET, f"{LOCAL_DATA}/Market-1501-v15.09.15")
    print("✅ Copy complete")
else:
    print("✅ Already copied")

# Verify
print(os.path.exists(f"{LOCAL_DATA}/Market-1501-v15.09.15/bounding_box_train"))

✅ Already copied
True


In [57]:
import os

# With dataset_dir = "", FastReID builds: root/Market-1501-v15.09.15/
# So DATA_ROOT just needs to contain Market-1501-v15.09.15/ directly

LOCAL_ROOT = "/content/datasets"
os.makedirs(LOCAL_ROOT, exist_ok=True)

target = f"{LOCAL_ROOT}/Market-1501-v15.09.15"
if not os.path.exists(target):
    os.symlink(
        "/content/drive/MyDrive/reid_project/Market-1501-v15.09.15",
        target
    )
    print(f"✅ Symlink created")
else:
    print("✅ Already exists")

# Verify all required folders
for folder in ["bounding_box_train", "bounding_box_test", "query"]:
    exists = os.path.exists(f"{target}/{folder}")
    print(f"  {folder}: {exists}")

✅ Symlink created
  bounding_box_train: True
  bounding_box_test: True
  query: True


In [58]:
# Cell 3: Training configuration
import os

# ── Paths ──────────────────────────────────────────────────
DRIVE_ROOT     = "/content/drive/MyDrive/reid_project"
DATA_ROOT      = "/content/datasets"                     # datasets live directly on Drive
MARKET_ROOT    = f"{DRIVE_ROOT}/Market-1501-v15.09.15"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────
BACKBONE       = "resnet50_ibn_a"
FEAT_DIM       = 2048
PRETRAINED     = True

# ── Training ───────────────────────────────────────────────
NUM_EPOCHS     = 60
WARMUP_EPOCHS  = 10
BATCH_SIZE     = 64
BASE_LR        = 0.00035
WEIGHT_DECAY   = 0.0005
NUM_WORKERS    = 2

# ── Loss ───────────────────────────────────────────────────
CIRCLE_SCALE   = 128
CIRCLE_MARGIN  = 0.25
LABEL_SMOOTH   = 0.1

# ── Augmentation ───────────────────────────────────────────
RANDOM_ERASING_PROB = 0.5
INPUT_SIZE     = (256, 128)  # H x W

# ── Reproducibility ────────────────────────────────────────
SEED           = 42

print("✅ Config set")
print(f"   Backbone  : {BACKBONE}")
print(f"   Epochs    : {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   LR        : {BASE_LR}")

✅ Config set
   Backbone  : resnet50_ibn_a
   Epochs    : 60
   Batch size: 64
   LR        : 0.00035


In [32]:
# Cell 4: Verify GPU
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected — check Runtime > Change runtime type > T4 GPU")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [68]:
# Cell 5: Build FastReID config object
import sys, os, importlib.util

# ── FastReID path fix ──────────────────────────────────────
def ensure_fastreid():
    if "fastreid" not in sys.modules:
        spec = importlib.util.spec_from_file_location(
            "fastreid",
            "/content/fast-reid/fastreid/__init__.py",
            submodule_search_locations=["/content/fast-reid/fastreid"]
        )
        fastreid = importlib.util.module_from_spec(spec)
        sys.modules["fastreid"] = fastreid
    if '/content/fast-reid' not in sys.path:
        sys.path.insert(0, '/content/fast-reid')

ensure_fastreid()
# ──────────────────────────────────────────────────────────

from fastreid.config import get_cfg

def build_reid_cfg():
    cfg = get_cfg()

    # Model
    cfg.MODEL.BACKBONE.NAME        = "build_resnet_backbone"
    cfg.MODEL.BACKBONE.DEPTH       = "50x"
    cfg.MODEL.BACKBONE.LAST_STRIDE = 1
    cfg.MODEL.BACKBONE.PRETRAIN    = PRETRAINED
    cfg.MODEL.BACKBONE.WITH_IBN    = True
    cfg.MODEL.HEADS.NAME           = "EmbeddingHead"
    cfg.MODEL.HEADS.NUM_CLASSES    = 751
    cfg.MODEL.HEADS.NECK_FEAT      = "before"
    cfg.MODEL.HEADS.POOL_LAYER = "GlobalAvgPool"

    # Loss
    cfg.MODEL.LOSSES.NAME          = ("CrossEntropyLoss", "CircleLoss")
    cfg.MODEL.LOSSES.CE.EPSILON    = LABEL_SMOOTH
    cfg.MODEL.LOSSES.CIRCLE.SCALE  = CIRCLE_SCALE
    cfg.MODEL.LOSSES.CIRCLE.MARGIN = CIRCLE_MARGIN

    # Dataset
    cfg.DATASETS.NAMES             = ("Market1501",)
    cfg.DATASETS.ROOT              = DATA_ROOT

    # Dataloader
    cfg.DATALOADER.NUM_WORKERS     = NUM_WORKERS
    cfg.DATALOADER.NUM_INSTANCE    = 4

    # Solver
    cfg.SOLVER.IMS_PER_BATCH       = BATCH_SIZE
    cfg.SOLVER.BASE_LR             = BASE_LR
    cfg.SOLVER.WEIGHT_DECAY        = WEIGHT_DECAY
    cfg.SOLVER.MAX_EPOCH           = NUM_EPOCHS
    cfg.SOLVER.SCHED               = "CosineAnnealingLR"
    cfg.SOLVER.OPT                 = "Adam"
    cfg.SOLVER.WARMUP_ITERS        = 2000    # warmup for 2000 iterations
    cfg.SOLVER.ETA_MIN_LR          = 1e-7    # minimum LR at end of cosine cycle
    cfg.SOLVER.LOG_PERIOD          = 20

    # Input
    cfg.INPUT.SIZE_TRAIN           = list(INPUT_SIZE)
    cfg.INPUT.SIZE_TEST            = list(INPUT_SIZE)
    cfg.INPUT.PROB                 = 0.5
    cfg.INPUT.REA.PROB             = RANDOM_ERASING_PROB

    # Output
    cfg.OUTPUT_DIR                 = "/content/reid_output"

    cfg.freeze()
    return cfg

cfg = build_reid_cfg()
print("✅ FastReID config built")

✅ FastReID config built


In [34]:
# Fix cell: patch Python 3.12 incompatibility in FastReID
file_path = "/content/fast-reid/fastreid/evaluation/testing.py"

with open(file_path, "r") as f:
    content = f.read()

# Fix the incompatible import
content = content.replace(
    "from collections import Mapping, OrderedDict",
    "from collections.abc import Mapping\nfrom collections import OrderedDict"
)

with open(file_path, "w") as f:
    f.write(content)

print("✅ Patched testing.py")

# Check if there are other files with the same issue
import subprocess
result = subprocess.run(
    "grep -r 'from collections import Mapping' /content/fast-reid/fastreid/",
    shell=True, capture_output=True, text=True
)
print("Remaining occurrences:", result.stdout if result.stdout else "None ✓")

✅ Patched testing.py
Remaining occurrences: None ✓


In [35]:
import subprocess, os

fastreid_root = "/content/fast-reid/fastreid"

# Find all files with the old collections imports
result = subprocess.run(
    f"grep -rl 'from collections import' {fastreid_root}",
    shell=True, capture_output=True, text=True
)
files = result.stdout.strip().split("\n")
print(f"Found {len(files)} files to check")

fixes = [
    ("from collections import Mapping, OrderedDict",
     "from collections.abc import Mapping\nfrom collections import OrderedDict"),
    ("from collections import Mapping",
     "from collections.abc import Mapping"),
    ("from collections import Callable",
     "from collections.abc import Callable"),
    ("from collections import Sequence",
     "from collections.abc import Sequence"),
    ("from collections import Iterable",
     "from collections.abc import Iterable"),
    ("from collections import MutableMapping",
     "from collections.abc import MutableMapping"),
]

patched = []
for fpath in files:
    if not fpath.strip():
        continue
    with open(fpath, "r") as f:
        content = f.read()
    original = content
    for old, new in fixes:
        content = content.replace(old, new)
    if content != original:
        with open(fpath, "w") as f:
            f.write(content)
        patched.append(fpath)

print(f"\nPatched {len(patched)} files:")
for p in patched:
    print(f"  ✓ {p.replace(fastreid_root, 'fastreid')}")

print("\n✅ All patches applied")

Found 19 files to check

Patched 0 files:

✅ All patches applied


In [45]:
# Create symlink so FastReID finds the data
# /content/datasets/Market-1501/Market-1501-v15.09.15 -> Drive
import os

market_link = "/content/datasets/Market-1501"
if not os.path.exists(market_link):
    os.symlink(
        "/content/drive/MyDrive/reid_project",
        market_link
    )
    print(f"✅ Symlink created: {market_link}")
else:
    print("✅ Symlink already exists")

# Verify
print(os.path.exists("/content/datasets/Market-1501/Market-1501-v15.09.15"))

✅ Symlink already exists
True


In [67]:
import inspect
from fastreid.solver.build import build_lr_scheduler
print(inspect.getsource(build_lr_scheduler))

def build_lr_scheduler(cfg, optimizer, iters_per_epoch):
    max_epoch = cfg.SOLVER.MAX_EPOCH - max(
        math.ceil(cfg.SOLVER.WARMUP_ITERS / iters_per_epoch), cfg.SOLVER.DELAY_EPOCHS)

    scheduler_dict = {}

    scheduler_args = {
        "MultiStepLR": {
            "optimizer": optimizer,
            # multi-step lr scheduler options
            "milestones": cfg.SOLVER.STEPS,
            "gamma": cfg.SOLVER.GAMMA,
        },
        "CosineAnnealingLR": {
            "optimizer": optimizer,
            # cosine annealing lr scheduler options
            "T_max": max_epoch,
            "eta_min": cfg.SOLVER.ETA_MIN_LR,
        },

    }

    scheduler_dict["lr_sched"] = getattr(lr_scheduler, cfg.SOLVER.SCHED)(
        **scheduler_args[cfg.SOLVER.SCHED])

    if cfg.SOLVER.WARMUP_ITERS > 0:
        warmup_args = {
            "optimizer": optimizer,

            # warmup options
            "warmup_factor": cfg.SOLVER.WARMUP_FACTOR,
            "warmup_iters": cfg.SOLVER.WARMUP_

In [69]:
# Cell 6: Train
import os
from fastreid.engine import DefaultTrainer

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

print("✅ Training complete")

[04/01 15:07:50 fastreid.engine.defaults]: Prepare training set
[04/01 15:07:51 fastreid.data.datasets.bases]: => Loaded Market1501 in csv format: 
| subset   | # ids   | # images   | # cameras   |
|:---------|:--------|:-----------|:------------|
| train    | 751     | 12936      | 6           |
[04/01 15:07:51 fastreid.data.build]: Using training sampler TrainingSampler
[04/01 15:07:51 fastreid.modeling.backbones.resnet]: Loading pretrained model from /root/.cache/torch/checkpoints/resnet50_ibn_a-d9d0bb7b.pth
[04/01 15:07:51 fastreid.modeling.backbones.resnet]: The checkpoint state_dict contains keys that are not used by the model:
  fc.{weight, bias}
[04/01 15:07:51 fastreid.engine.defaults]: Model:
Baseline(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride

/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))


[04/01 15:07:52 fastreid.utils.checkpoint]: No checkpoint found. Training model from scratch
[04/01 15:07:52 fastreid.engine.train_loop]: Starting training from epoch 0
[04/01 15:11:21 fastreid.utils.events]:  eta: 1:41:13  epoch/iter: 0/199  total_loss: 969.9  loss_cls: 6.659  loss_circle: 963.2  time: 0.5105  data_time: 0.0014  lr: 6.63e-05  max_mem: 4822M
[04/01 15:11:22 fastreid.utils.events]:  eta: 1:41:04  epoch/iter: 0/201  total_loss: 970.5  loss_cls: 6.659  loss_circle: 963.9  time: 0.5104  data_time: 0.0013  lr: 6.67e-05  max_mem: 4822M
[04/01 15:13:03 fastreid.utils.events]:  eta: 1:39:04  epoch/iter: 1/399  total_loss: 977.1  loss_cls: 6.659  loss_circle: 970.5  time: 0.5106  data_time: 0.0008  lr: 9.78e-05  max_mem: 4822M
[04/01 15:13:05 fastreid.utils.events]:  eta: 1:39:02  epoch/iter: 1/403  total_loss: 975  loss_cls: 6.659  loss_circle: 968.3  time: 0.5107  data_time: 0.0012  lr: 9.85e-05  max_mem: 4822M
[04/01 15:14:46 fastreid.utils.events]:  eta: 1:37:40  epoch/iter

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[04/01 15:44:47 fastreid.evaluation.reid_evaluation]: >>> done with reid evaluation cython tool. Compilation time: 24.816 seconds
[04/01 15:44:47 fastreid.evaluation.evaluator]: Start inference on 19281 images


...

[04/01 15:46:30 fastreid.evaluation.evaluator]: Inference done 11/302. 0.1493 s / batch. ETA=0:01:10
[04/01 15:47:06 fastreid.evaluation.evaluator]: Inference done 53/302. 0.1521 s / batch. ETA=0:03:13
[04/01 15:47:51 fastreid.evaluation.evaluator]: Inference done 61/302. 0.1547 s / batch. ETA=0:05:51
[04/01 15:48:35 fastreid.evaluation.evaluator]: Inference done 69/302. 0.1563 s / batch. ETA=0:07:40
[04/01 15:49:19 fastreid.evaluation.evaluator]: Inference done 77/302. 0.1574 s / batch. ETA=0:08:51
[04/01 15:50:03 fastreid.evaluation.evaluator]: Inference done 85/302. 0.1588 s / batch. ETA=0:09:41
[04/01 15:50:49 fastreid.evaluation.evaluator]: Inference done 93/302. 0.1602 s / batch. ETA=0:10:17
[04/01 15:51:33 fastreid.evaluation.evaluator]: Inference done 101/302. 0.1613 s / batch. ETA=0:10:36
[04/01 15:52:17 fastreid.evaluation.evaluator]: Inference done 109/302. 0.1622 s / batch. ETA=0:10:46
[04/01 15:53:00 fastreid.evaluation.evaluator]: Inference done 117/302. 0.1626 s / batch.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. 

[04/01 16:45:16 fastreid.evaluation.evaluator]: Inference done 11/302. 0.1514 s / batch. ETA=0:01:35
[04/01 16:45:46 fastreid.evaluation.evaluator]: Inference done 77/302. 0.1553 s / batch. ETA=0:01:40
[04/01 16:46:22 fastreid.evaluation.evaluator]: Inference done 85/302. 0.1566 s / batch. ETA=0:03:05
[04/01 16:46:53 fastreid.evaluation.evaluator]: Inference done 180/302. 0.1585 s / batch. ETA=0:01:09
[04/01 16:47:23 fastreid.evaluation.evaluator]: Inference done 273/302. 0.1577 s / batch. ETA=0:00:14
[04/01 16:47:32 fastreid.evaluation.evaluator]: Total inference time: 0:02:18.380055 (0.465926 s / batch per device, on 1 devices)
[04/01 16:47:32 fastreid.evaluation.evaluator]: Total inference pure compute time: 0:00:46 (0.157014 s / batch per device, on 1 devices)
[04/01 16:47:39 fastreid.engine.defaults]: Evaluation results for Market1501 in csv format:
[04/01 16:47:39 fastreid.evaluation.testing]: Evaluation results in csv format: 
| Dataset    | Rank-1   | Rank-5   | Rank-10   | mAP

/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be t

[04/01 17:22:20 fastreid.evaluation.evaluator]: Inference done 11/302. 0.1563 s / batch. ETA=0:01:36
[04/01 17:22:50 fastreid.evaluation.evaluator]: Inference done 77/302. 0.1555 s / batch. ETA=0:01:40
[04/01 17:23:20 fastreid.evaluation.evaluator]: Inference done 144/302. 0.1573 s / batch. ETA=0:01:10
[04/01 17:23:50 fastreid.evaluation.evaluator]: Inference done 209/302. 0.1570 s / batch. ETA=0:00:42
[04/01 17:24:21 fastreid.evaluation.evaluator]: Inference done 281/302. 0.1569 s / batch. ETA=0:00:09
[04/01 17:24:28 fastreid.evaluation.evaluator]: Total inference time: 0:02:09.750248 (0.436870 s / batch per device, on 1 devices)
[04/01 17:24:28 fastreid.evaluation.evaluator]: Total inference pure compute time: 0:00:46 (0.156676 s / batch per device, on 1 devices)
[04/01 17:24:37 fastreid.engine.defaults]: Evaluation results for Market1501 in csv format:
[04/01 17:24:37 fastreid.evaluation.testing]: Evaluation results in csv format: 
| Dataset    | Rank-1   | Rank-5   | Rank-10   | mA

In [70]:
# Cell 7: Copy best checkpoint to Google Drive
import shutil, glob

output_dir  = cfg.OUTPUT_DIR
best_ckpt   = f"{output_dir}/model_best.pth"
checkpoints = sorted(glob.glob(f"{output_dir}/model_*.pth"))

if not os.path.exists(best_ckpt):
    best_ckpt = checkpoints[-1] if checkpoints else None

if best_ckpt:
    dest = f"{DRIVE_CKPT_DIR}/reid_best.pth"
    shutil.copy(best_ckpt, dest)
    print(f"✅ Best checkpoint saved to Drive: {dest}")
else:
    print("❌ No checkpoint found — check training output")

✅ Best checkpoint saved to Drive: /content/drive/MyDrive/reid_project/checkpoints/reid_best.pth


In [74]:
# Cell 8: Evaluate on Market-1501 test split
from fastreid.evaluation import ReidEvaluator
from fastreid.data import build_reid_test_loader

cfg_test = cfg.clone()
cfg_test.defrost()
cfg_test.DATASETS.NAMES = ("Market1501",)
cfg_test.freeze()

test_loader, num_query = build_reid_test_loader(
    cfg_test,
    dataset_name="Market1501"
)
evaluator = ReidEvaluator(cfg_test, num_query)

results = DefaultTrainer.test(cfg_test, trainer.model, evaluators=[evaluator])
print("\n── Market-1501 Results ──────────────────")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

[04/01 17:33:55 fastreid.data.datasets.bases]: => Loaded Market1501 in csv format: 
| subset   | # ids   | # images   | # cameras   |
|:---------|:--------|:-----------|:------------|
| query    | 750     | 3368       | 6           |
| gallery  | 751     | 15913      | 6           |


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


TypeError: DefaultTrainer.test() got an unexpected keyword argument 'evaluators'

In [73]:
from fastreid.evaluation import ReidEvaluator
from fastreid.data import build_reid_test_loader

cfg_test = cfg.clone()
cfg_test.defrost()
cfg_test.DATASETS.NAMES = ("Market1501",)
cfg_test.freeze()

test_loader, num_query = build_reid_test_loader(cfg_test)
evaluator = ReidEvaluator(cfg_test, num_query)

results = DefaultTrainer.test(cfg_test, trainer.model, evaluators=[evaluator])

print("\n── Market-1501 Results ──────────────────")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

AssertionError: dataset_name must be explicitly passed in when test_set is not provided

In [75]:
cfg_test = cfg.clone()
cfg_test.defrost()
cfg_test.DATASETS.TESTS = ("Market1501",)   # use TESTS, not NAMES
cfg_test.freeze()

results = DefaultTrainer.test(cfg_test, trainer.model)

print("\n── Market-1501 Results ──────────────────")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

[04/01 17:35:26 fastreid.engine.defaults]: Prepare testing set
[04/01 17:35:34 fastreid.data.datasets.bases]: => Loaded Market1501 in csv format: 
| subset   | # ids   | # images   | # cameras   |
|:---------|:--------|:-----------|:------------|
| query    | 750     | 3368       | 6           |
| gallery  | 751     | 15913      | 6           |
[04/01 17:35:34 fastreid.evaluation.evaluator]: Start inference on 19281 images


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/fast-reid/fastreid/data/transforms/functional.py:46: UserWarning: TypedStorage is deprecated. 

[04/01 17:36:09 fastreid.evaluation.evaluator]: Inference done 11/302. 0.1476 s / batch. ETA=0:01:04
[04/01 17:36:39 fastreid.evaluation.evaluator]: Inference done 106/302. 0.1506 s / batch. ETA=0:01:00
[04/01 17:37:09 fastreid.evaluation.evaluator]: Inference done 205/302. 0.1525 s / batch. ETA=0:00:29
[04/01 17:37:39 fastreid.evaluation.evaluator]: Inference done 302/302. 0.1541 s / batch. ETA=0:00:00
[04/01 17:37:39 fastreid.evaluation.evaluator]: Total inference time: 0:01:31.496564 (0.308069 s / batch per device, on 1 devices)
[04/01 17:37:39 fastreid.evaluation.evaluator]: Total inference pure compute time: 0:00:45 (0.154105 s / batch per device, on 1 devices)
[04/01 17:37:46 fastreid.engine.defaults]: Evaluation results for Market1501 in csv format:
[04/01 17:37:46 fastreid.evaluation.testing]: Evaluation results in csv format: 
| Dataset    | Rank-1   | Rank-5   | Rank-10   | mAP   | mINP   | metric   |
|:-----------|:---------|:---------|:----------|:------|:-------|:---------

In [76]:
# Cell 9: Instructions — download checkpoint to M1
print("""
╔══════════════════════════════════════════════════════════╗
║         Training complete — next steps on M1             ║
╠══════════════════════════════════════════════════════════╣
║  1. Open Google Drive in browser                         ║
║  2. Navigate to: MyDrive/reid_project/checkpoints/       ║
║  3. Download: reid_best.pth                              ║
║  4. Move to your project:                                ║
║     mv ~/Downloads/reid_best.pth                         ║
║        ~/Desktop/Portfolio\ Projects/                    ║
║           person-reidentification/checkpoints/           ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         Training complete — next steps on M1             ║
╠══════════════════════════════════════════════════════════╣
║  1. Open Google Drive in browser                         ║
║  2. Navigate to: MyDrive/reid_project/checkpoints/       ║
║  3. Download: reid_best.pth                              ║
║  4. Move to your project:                                ║
║     mv ~/Downloads/reid_best.pth                         ║
║        ~/Desktop/Portfolio\ Projects/                    ║
║           person-reidentification/checkpoints/           ║
╚══════════════════════════════════════════════════════════╝



<>:11: SyntaxWarning: invalid escape sequence '\ '
<>:11: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4056/4202663471.py:11: SyntaxWarning: invalid escape sequence '\ '
  ║        ~/Desktop/Portfolio\ Projects/                    ║
